<a href="https://colab.research.google.com/github/ZeroOneThree013/BTC_AI_Trading_Strategy/blob/main/BTC_AI_Trading_Strategy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🪙 AI 預測比特幣價格波動與交易策略系統
> **環境**：Google Colab (T4 GPU) | **前端**：Gradio 4.x

## 📋 執行順序
1. **Cell 1**：安裝套件
2. **Cell 2**：匯入套件與全域狀態
3. **Cell 3**：抓取資料
4. **Cell 4**：特徵工程
5. **Cell 5**：訓練模型（LSTM + XGBoost + SHAP）
6. **Cell 6**：回測引擎
7. **Cell 7**：啟動 Gradio Dashboard

### Cell 1：安裝套件

In [8]:
!pip install -q gradio plotly pandas-ta xgboost shap requests

### Cell 2：匯入套件與全域狀態

In [1]:
import requests, time, pickle, warnings
import numpy as np
import pandas as pd
import pandas_ta as ta
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import gradio as gr
import shap
warnings.filterwarnings("ignore")

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from xgboost import XGBClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

np.random.seed(42)
tf.random.set_seed(42)

STATE = dict(df_raw=None, df_features=None, model_lstm=None, model_xgb=None,
             scaler=None, feature_cols=None, df_pred=None,
             df_backtest_lstm=None, df_backtest_xgb=None,
             bt_metrics_lstm=None,  bt_metrics_xgb=None,
             shap_values=None, X_shap=None, eval_results=None,
             trade_log_lstm=[], trade_log_xgb=[])
print("✅ 套件載入完成")

✅ 套件載入完成


### Cell 3：資料抓取（Yahoo Finance 小時線，730 天）

In [35]:
!pip install -q yfinance ta

import yfinance as yf
import ta as ta_lib
from datetime import datetime, timedelta

def fetch_btc():
    print("📡 抓取 BTC-USD 小時線（730 天）...")
    df = yf.download("BTC-USD", period="730d", interval="1h", progress=False)

    # 處理新版 yfinance MultiIndex
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    if df.empty:
        raise ValueError("抓取失敗，請檢查網路連線")

    df = df.reset_index()
    df = df.rename(columns={"Datetime": "date", "Open": "open", "High": "high",
                             "Low": "low", "Close": "close", "Volume": "volume"})
    df["date"] = pd.to_datetime(df["date"])
    df = df[["date","open","high","low","close","volume"]].copy()
    for c in ["open","high","low","close","volume"]:
        df[c] = pd.to_numeric(df[c])
    return df.drop_duplicates("date").sort_values("date").reset_index(drop=True)

def load_data():
    try:
        df = fetch_btc()
        df = df.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)
        STATE["df_raw"] = df
        miss = df.isnull().mean().mean()
        print(f"✅ 完成：{len(df)} 筆 | 缺漏率：{miss:.2%} | {df['date'].min()} ~ {df['date'].max()}")
    except Exception as e:
        print(f"❌ 資料載入失敗：{e}")

load_data()

📡 抓取 BTC-USD 小時線（730 天）...
✅ 完成：17348 筆 | 缺漏率：0.00% | 2024-04-28 00:00:00+00:00 ~ 2026-04-27 07:00:00+00:00


### Cell 4：特徵工程（ta.add_all_ta_features，80+ 個技術指標）

In [36]:
SEQ_LEN = 24

def build_features(df):
    d = df.copy()

    # 計算所有技術指標
    d = ta_lib.add_all_ta_features(
        d, open="open", high="high", low="low",
        close="close", volume="volume", fillna=True
    )

    # 加入報酬率特徵
    d["close_ret"] = d["close"].pct_change()
    d["high_ret"]  = d["high"].pct_change()
    d["low_ret"]   = d["low"].pct_change()
    d["vol_ret"]   = d["volume"].pct_change()

    # label
    d["label"] = (d["close"].shift(-1) > d["close"]).astype(int)

    # 清除 inf 與 NaN
    d = d.replace([np.inf, -np.inf], np.nan)
    before = len(d)
    d = d.dropna().reset_index(drop=True)
    print(f"   移除 NaN：{before - len(d)} 筆，剩餘 {len(d)} 筆")
    return d


def split_scale(df, seq_len=SEQ_LEN):
    # 自動擷取所有特徵欄位
    exclude = ["date","label","open","high","low","close","volume"]
    FEAT    = [c for c in df.columns if c not in exclude]

    n, t, v = len(df), int(len(df)*.70), int(len(df)*.85)
    sc  = MinMaxScaler()
    X   = df[FEAT].values.astype(float)
    X[:t] = sc.fit_transform(X[:t])
    X[t:] = sc.transform(X[t:])
    y = df["label"].values

    def seq(X, y, s, e):
        Xs, ys = [], []
        for i in range(s, e - seq_len):
            Xs.append(X[i:i+seq_len]); ys.append(y[i+seq_len])
        return np.array(Xs), np.array(ys)

    Xtr,ytr = seq(X,y,0,t)
    Xva,yva = seq(X,y,t,v)
    Xte,yte = seq(X,y,v,n)

    with open("/content/scaler.pkl","wb") as f: pickle.dump(sc,f)
    STATE.update(scaler=sc, feature_cols=FEAT, df_features=df)
    print(f"✅ 特徵完成 | 特徵數：{len(FEAT)} | Train:{Xtr.shape} Val:{Xva.shape} Test:{Xte.shape}")
    return Xtr,ytr,Xva,yva,Xte,yte,FEAT


print("🔧 執行特徵工程...")
df_feat = build_features(STATE["df_raw"])
Xtr,ytr,Xva,yva,Xte,yte,FEAT = split_scale(df_feat)

🔧 執行特徵工程...
   移除 NaN：8771 筆，剩餘 8577 筆
✅ 特徵完成 | 特徵數：90 | Train:(5979, 24, 90) Val:(1263, 24, 90) Test:(1263, 24, 90)


### Cell 5：模型訓練（LSTM + SHAP）

In [37]:
from tensorflow.keras.layers import BatchNormalization

def train_lstm(Xtr,ytr,Xva,yva):
    print("🧠 訓練 LSTM...")
    m = Sequential([
        LSTM(64, input_shape=(Xtr.shape[1], Xtr.shape[2])),
        BatchNormalization(),
        Dropout(0.2),
        Dense(1, activation="sigmoid")
    ])
    m.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    m.fit(Xtr, ytr, validation_data=(Xva,yva), epochs=100, batch_size=64, verbose=1,
          callbacks=[EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
                     ModelCheckpoint("/content/model_lstm.keras", save_best_only=True)])
    STATE["model_lstm"] = m
    print("✅ LSTM 完成")
    return m


def calc_shap(m, Xtr, Xte):
    print("🔍 計算 SHAP（GradientExplainer）...")
    # 用訓練集前 100 筆作為背景資料
    background = Xtr[:100]
    explainer  = shap.GradientExplainer(m, background)
    # 取測試集前 200 筆計算 SHAP
    sv = explainer.shap_values(Xte[:200])
    # sv shape: (200, 24, 90) → 對時間步取平均，得到 (200, 90)
    sv_mean = np.abs(sv[0]).mean(axis=1)
    STATE.update(shap_values=sv_mean, X_shap=Xte[:200])
    print("✅ SHAP 完成")


def evaluate(Xte, yte):
    res = {}
    if STATE["model_lstm"]:
        prob = STATE["model_lstm"].predict(Xte).flatten()
        p    = (prob > .5).astype(int)
        res["LSTM"] = {"accuracy": accuracy_score(yte, p),
                       "f1":       f1_score(yte, p),
                       "auc":      roc_auc_score(yte, prob),
                       "pred":     p, "prob": prob}
    for n,r in res.items():
        print(f"  {n} → Acc:{r['accuracy']:.4f} F1:{r['f1']:.4f} AUC:{r['auc']:.4f}")
    return res


# 執行
train_lstm(Xtr, ytr, Xva, yva)
calc_shap(STATE["model_lstm"], Xtr, Xte)
er = evaluate(Xte, yte)

# 儲存預測
sl = STATE["df_features"].iloc[-len(yte):].copy().reset_index(drop=True)
sl["y_true"]    = yte
sl["pred_lstm"] = er["LSTM"]["pred"]
sl["prob_lstm"] = er["LSTM"]["prob"]
sl["pred_xgb"]  = sl["pred_lstm"]
sl["prob_xgb"]  = sl["prob_lstm"]

STATE.update(df_pred=sl, eval_results=er)
print("\n✅ 完成，可繼續執行 Cell 6")

🧠 訓練 LSTM...
Epoch 1/100
94/94 ━━━━━━━━━━━━━━━━━━━━ 9s 56ms/step - accuracy: 0.5081 - loss: 0.7442 - val_accuracy: 0.4941 - val_loss: 0.7100
Epoch 2/100
94/94 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.5073 - loss: 0.7294 - val_accuracy: 0.4893 - val_loss: 0.6963
Epoch 3/100
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.5098 - loss: 0.7135 - val_accuracy: 0.4964 - val_loss: 0.6958
Epoch 4/100
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.5156 - loss: 0.7099 - val_accuracy: 0.4949 - val_loss: 0.6975
Epoch 5/100
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.5051 - loss: 0.7060 - val_accuracy: 0.4980 - val_loss: 0.6936
Epoch 6/100
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.5258 - loss: 0.6996 - val_accuracy: 0.5123 - val_loss: 0.6958
Epoch 7/100
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.5210 - loss: 0.6968 - val_accuracy: 0.4980 - val_loss: 0.6947
Epoch 8/100
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.5183 - loss: 0.6964 - val_accu

### Cell 6：回測引擎（多策略自動選擇）

In [42]:
def run_backtest(df_pred, model_type="LSTM", capital=10000.0, fee=0.001,
                 buy_thresh=None, sell_thresh=None, strategy_name=""):
    if capital <= 0: raise ValueError("初始資金必須為正數")
    if fee < 0:      raise ValueError("手續費不可為負數")

    pc  = "prob_lstm" if model_type == "LSTM" else "prob_xgb"
    df  = df_pred.copy().reset_index(drop=True)
    cash, btc = capital, 0.0
    eq, sig, log = [], [], []
    trade_returns = []
    buy_price = None

    for row in df.itertuples():
        p, price = getattr(row, pc), row.close
        if p > buy_thresh and btc == 0:
            btc = (cash * (1 - fee)) / price; cash = 0
            buy_price = price
            sig.append("Buy")
            log.append({"date": row.date, "action": "Buy", "price": price})
        elif p < sell_thresh and btc > 0:
            cash = btc * price * (1 - fee); btc = 0
            sig.append("Sell")
            log.append({"date": row.date, "action": "Sell", "price": price})
            if buy_price:
                trade_returns.append((price - buy_price) / buy_price)
                buy_price = None
        else:
            sig.append("Hold")
        eq.append(cash + btc * price)

    df["signal"]     = sig
    df["equity"]     = eq
    df["bah_equity"] = capital * df["close"] / df["close"].iloc[0]

    ret        = pd.Series(eq).pct_change().dropna()
    sharpe     = (ret.mean() / ret.std() * (252 * 24) ** 0.5) if ret.std() > 0 else 0
    mdd        = ((pd.Series(eq) - pd.Series(eq).cummax()) / pd.Series(eq).cummax()).min()
    win_rate   = sum(1 for r in trade_returns if r > 0) / len(trade_returns) if trade_returns else 0
    bah_return = (df["bah_equity"].iloc[-1] - capital) / capital

    m = {
        "strategy":     strategy_name,
        "total_return": (eq[-1] - capital) / capital,
        "bah_return":   bah_return,
        "sharpe":       sharpe,
        "mdd":          mdd,
        "trade_count":  len(trade_returns),
        "win_rate":     win_rate,
        "buy_thresh":   buy_thresh,
        "sell_thresh":  sell_thresh,
        "df":           df,
        "log":          log,
    }
    return m


def auto_select_strategy(df_pred, model_type="LSTM", capital=10000.0, fee=0.001):
    pc = "prob_lstm" if model_type == "LSTM" else "prob_xgb"
    df = df_pred.copy()

    # ── 定義策略 ──────────────────────────────────────
    strategies = [
        {
            "name":        "保守策略（高閾值空手）",
            "description": "只在信心極高時進場，大部分時間空手",
            "buy_thresh":  df[pc].quantile(0.90),
            "sell_thresh": df[pc].quantile(0.10),
        },
        {
            "name":        "穩健策略（平衡進出）",
            "description": "適度進出，兼顧報酬與風險",
            "buy_thresh":  df[pc].quantile(0.80),
            "sell_thresh": df[pc].quantile(0.20),
        },
        {
            "name":        "積極策略（低閾值常交易）",
            "description": "頻繁進出，追求更多交易機會",
            "buy_thresh":  df[pc].quantile(0.65),
            "sell_thresh": df[pc].quantile(0.35),
        },
        {
            "name":        "趨勢追蹤策略",
            "description": "只做 Buy，不主動 Sell，靠趨勢獲利",
            "buy_thresh":  df[pc].quantile(0.80),
            "sell_thresh": df[pc].quantile(0.01),
        },
        {
            "name":        "Buy-and-Hold 基準",
            "description": "買入後持有不動，作為對照組",
            "buy_thresh":  df[pc].quantile(0.01),
            "sell_thresh": df[pc].quantile(0.001),
        },
    ]

    # ── 執行所有策略回測 ───────────────────────────────
    print("=" * 60)
    print(f"{'策略':<22} {'報酬率':>8} {'Sharpe':>8} {'MDD':>8} {'勝率':>8} {'交易':>6}")
    print("=" * 60)

    results = []
    for s in strategies:
        m = run_backtest(df_pred, model_type=model_type, capital=capital, fee=fee,
                         buy_thresh=s["buy_thresh"], sell_thresh=s["sell_thresh"],
                         strategy_name=s["name"])
        m["description"] = s["description"]
        results.append(m)
        print(f"{s['name']:<22} {m['total_return']:>7.2%} {m['sharpe']:>8.2f} "
              f"{m['mdd']:>7.2%} {m['win_rate']:>7.2%} {m['trade_count']:>6}")

    print("=" * 60)
    print(f"Buy-and-Hold 報酬率：{results[0]['bah_return']:.2%}")

    # ── 自動選擇最佳策略（排除 Buy-and-Hold 基準）─────
    valid = [r for r in results
             if r["trade_count"] > 0 and r["strategy"] != "Buy-and-Hold 基準"]

    if not valid:
        print("\n⚠️ 所有策略均無交易，請重新訓練模型")
        return None

    # 綜合評分：Sharpe(60%) + 報酬率(40%)，MDD 超過 -50% 直接排除
    def score(r):
        if r["mdd"] < -0.50:
            return -999
        return r["sharpe"] * 0.6 + r["total_return"] * 0.4

    best = max(valid, key=score)

    print(f"\n🏆 自動選擇最佳策略：【{best['strategy']}】")
    print(f"   📋 說明：{best['description']}")
    print(f"   💰 總報酬率：{best['total_return']:.2%}  vs  Buy-and-Hold：{best['bah_return']:.2%}")
    print(f"   📊 Sharpe：{best['sharpe']:.2f} | MDD：{best['mdd']:.2%} | "
          f"勝率：{best['win_rate']:.2%} | 交易次數：{best['trade_count']} 筆")

    # 儲存最佳策略至 STATE
    key = model_type.lower()
    STATE[f"df_backtest_{key}"] = best["df"]
    STATE[f"bt_metrics_{key}"]  = best
    STATE[f"trade_log_{key}"]   = best["log"]
    STATE["all_strategy_results"] = results

    return best


if STATE["df_pred"] is not None:
    auto_select_strategy(STATE["df_pred"], model_type="LSTM")
else:
    print("⚠️ 請先完成模型訓練（Cell 5）")

策略                          報酬率   Sharpe      MDD       勝率     交易
保守策略（高閾值空手）            -23.10%    -2.98 -24.77%  33.33%     21
穩健策略（平衡進出）             -17.11%    -1.79 -30.89%  42.86%     35
積極策略（低閾值常交易）           -13.94%    -1.40 -26.38%  44.44%     63
趨勢追蹤策略                  -7.75%    -0.42 -29.42%  16.67%      6
Buy-and-Hold 基準        -14.67%    -1.01 -35.13%   0.00%      2
Buy-and-Hold 報酬率：-15.27%

🏆 自動選擇最佳策略：【趨勢追蹤策略】
   📋 說明：只做 Buy，不主動 Sell，靠趨勢獲利
   💰 總報酬率：-7.75%  vs  Buy-and-Hold：-15.27%
   📊 Sharpe：-0.42 | MDD：-29.42% | 勝率：16.67% | 交易次數：6 筆


### Cell 7：Gradio Dashboard

In [43]:
# ── 圖表函式 ────────────────────────────────────────

def fig_overview():
    df = STATE["df_raw"]
    if df is None:
        return go.Figure().add_annotation(text="資料載入失敗，請重新執行 Cell 3", showarrow=False)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df["date"].astype(str), y=df["close"],
                             name="BTC 收盤價", line=dict(color="#F7931A")))
    fig.update_layout(title="BTC 收盤價走勢", template="plotly_dark",
                      height=430, yaxis_title="Price (USDT)")
    return fig


def fig_predict():
    df = STATE["df_pred"]
    if df is None:
        return go.Figure().add_annotation(text="請先完成模型訓練（Cell 5）", showarrow=False)
    x   = df["date"].astype(str)
    fig = make_subplots(rows=2, cols=1,
                        subplot_titles=("預測 vs 實際漲跌", "K 線圖與交易訊號"),
                        row_heights=[0.35, 0.65])
    fig.add_trace(go.Scatter(x=x, y=df["y_true"],    name="實際",  line=dict(color="#00D4AA")), row=1, col=1)
    fig.add_trace(go.Scatter(x=x, y=df["pred_lstm"], name="預測",  line=dict(color="#FF6B6B", dash="dash")), row=1, col=1)
    fig.add_trace(go.Candlestick(x=x, open=df["open"], high=df["high"],
                                 low=df["low"], close=df["close"], name="K線"), row=2, col=1)
    dbt = STATE.get("df_backtest_lstm")
    if dbt is not None:
        b = dbt[dbt["signal"] == "Buy"]
        s = dbt[dbt["signal"] == "Sell"]
        fig.add_trace(go.Scatter(x=b["date"].astype(str), y=b["close"], mode="markers",
                                 name="Buy",  marker=dict(color="#00D4AA", symbol="triangle-up",   size=10)), row=2, col=1)
        fig.add_trace(go.Scatter(x=s["date"].astype(str), y=s["close"], mode="markers",
                                 name="Sell", marker=dict(color="#FF6B6B", symbol="triangle-down", size=10)), row=2, col=1)
    fig.update_layout(template="plotly_dark", height=580)
    return fig


def fig_backtest_by_strategy(strategy_name):
    """依策略名稱產生資金曲線圖"""
    all_results = STATE.get("all_strategy_results")
    if not all_results:
        return go.Figure().add_annotation(text="請先執行回測（Cell 6）", showarrow=False)

    # 找到對應策略
    target = next((r for r in all_results if r["strategy"] == strategy_name), None)
    if target is None:
        return go.Figure().add_annotation(text=f"找不到策略：{strategy_name}", showarrow=False)

    df  = target["df"]
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df["date"].astype(str), y=df["equity"],
                             name="AI 策略", line=dict(color="#F7931A")))
    fig.add_trace(go.Scatter(x=df["date"].astype(str), y=df["bah_equity"],
                             name="Buy-and-Hold", line=dict(color="#6C8EBF", dash="dot")))
    fig.update_layout(title=f"資金曲線：{strategy_name}",
                      template="plotly_dark", height=400, yaxis_title="資產 (USDT)")
    return fig


def fig_shap():
    if STATE["shap_values"] is None:
        return go.Figure().add_annotation(text="請先完成模型訓練（Cell 5）", showarrow=False)
    sv  = STATE["shap_values"]
    fc  = STATE["feature_cols"]
    imp = np.abs(sv).mean(axis=0)
    idx = np.argsort(imp)[-10:]
    fig = go.Figure(go.Bar(x=imp[idx], y=[fc[i] for i in idx],
                           orientation="h", marker_color="#F7931A"))
    fig.update_layout(title="SHAP 特徵重要性（前 10）", template="plotly_dark",
                      height=400, xaxis_title="平均 |SHAP|")
    return fig


# ── Gradio 互動函式 ─────────────────────────────────

def act_overview():
    df = STATE["df_raw"]
    if df is None:
        return fig_overview(), "❌ 資料載入失敗，請重新執行 Cell 3"
    miss = df.isnull().mean().mean()
    return fig_overview(), (f"📊 資料筆數：{len(df)} 筆\n"
                            f"📅 時間範圍：{df['date'].min()} ~ {df['date'].max()}\n"
                            f"⚠️ 缺漏率：{miss:.2%}")


def act_predict():
    if STATE["df_pred"] is None:
        return fig_predict(), "❌ 模型尚未訓練，請先執行 Cell 5"
    er = STATE.get("eval_results", {})
    if "LSTM" not in er:
        return fig_predict(), "❌ LSTM 尚未訓練"
    r = er["LSTM"]
    return fig_predict(), (f"🎯 Accuracy：{r['accuracy']:.4f}\n"
                           f"📈 F1 Score：{r['f1']:.4f}\n"
                           f"📊 AUC-ROC：{r['auc']:.4f}")


def get_strategy_choices():
    """取得所有策略名稱，最佳策略加上 🏆 標記"""
    all_results = STATE.get("all_strategy_results", [])
    best        = STATE.get("bt_metrics_lstm", {})
    best_name   = best.get("strategy", "")
    choices     = []
    for r in all_results:
        label = f"🏆 {r['strategy']}" if r["strategy"] == best_name else r["strategy"]
        choices.append(label)
    return choices


def act_load_strategies():
    """載入所有策略，更新下拉選單並顯示策略總表"""
    all_results = STATE.get("all_strategy_results")
    if not all_results:
        return gr.update(choices=[], value=None), "❌ 請先執行回測（Cell 6）"

    best_name = STATE.get("bt_metrics_lstm", {}).get("strategy", "")
    choices   = get_strategy_choices()

    # 產生策略總表
    lines = ["策略比較總表\n" + "=" * 55]
    for r in all_results:
        tag  = " 🏆 最佳" if r["strategy"] == best_name else ""
        lines.append(
            f"{r['strategy']}{tag}\n"
            f"  報酬率：{r['total_return']:.2%} | Sharpe：{r['sharpe']:.2f} | "
            f"MDD：{r['mdd']:.2%} | 勝率：{r['win_rate']:.2%} | 交易：{r['trade_count']} 筆"
        )
    lines.append(f"\nBuy-and-Hold 報酬率：{all_results[0]['bah_return']:.2%}")

    default = next((c for c in choices if "🏆" in c), choices[0] if choices else None)
    return gr.update(choices=choices, value=default), "\n".join(lines)


def act_select_strategy(strategy_label):
    """選擇策略後更新圖表與績效"""
    if not strategy_label:
        return go.Figure(), ""

    # 移除 🏆 標記還原原始名稱
    strategy_name = strategy_label.replace("🏆 ", "")
    all_results   = STATE.get("all_strategy_results", [])
    target        = next((r for r in all_results if r["strategy"] == strategy_name), None)

    if target is None:
        return go.Figure(), f"❌ 找不到策略：{strategy_name}"

    best_name = STATE.get("bt_metrics_lstm", {}).get("strategy", "")
    tag       = "🏆 自動選擇最佳策略\n" if strategy_name == best_name else ""
    txt = (f"{tag}"
           f"📋 策略：{target['strategy']}\n"
           f"💰 總報酬率：{target['total_return']:.2%}  vs  Buy-and-Hold：{target['bah_return']:.2%}\n"
           f"📊 Sharpe：{target['sharpe']:.2f}\n"
           f"📉 最大回撤：{target['mdd']:.2%}\n"
           f"🏅 勝率：{target['win_rate']:.2%}\n"
           f"🔄 交易次數：{target['trade_count']} 筆")

    return fig_backtest_by_strategy(strategy_name), txt


def act_shap():
    if STATE["shap_values"] is None:
        return go.Figure(), "❌ 請先完成模型訓練（Cell 5）"
    return fig_shap(), "✅ SHAP 特徵重要性（基於 LSTM GradientExplainer，對時間步取平均）"


# ── Gradio UI ───────────────────────────────────────

with gr.Blocks(theme=gr.themes.Base(), title="BTC AI 交易策略") as demo:
    gr.Markdown("# 🪙 AI 預測比特幣價格波動與交易策略系統")
    gr.Markdown("> 請依序完成 Cell 1–6 後，在此 Dashboard 進行互動分析")

    with gr.Tab("📊 資料總覽"):
        btn1 = gr.Button("🔄 載入資料總覽", variant="primary")
        p1   = gr.Plot()
        t1   = gr.Textbox(label="資料摘要", lines=3)
        btn1.click(act_overview, outputs=[p1, t1])

    with gr.Tab("🤖 模型預測"):
        btn2 = gr.Button("🔍 產生預測", variant="primary")
        p2   = gr.Plot()
        t2   = gr.Textbox(label="模型績效", lines=3)
        btn2.click(act_predict, outputs=[p2, t2])

    with gr.Tab("📈 回測報告"):
        btn3    = gr.Button("📊 載入所有策略", variant="primary")
        dd3     = gr.Dropdown(choices=[], label="選擇策略（🏆 = 自動最佳）", interactive=True)
        t3_all  = gr.Textbox(label="策略比較總表", lines=12)
        p3      = gr.Plot()
        t3      = gr.Textbox(label="選定策略績效", lines=7)
        btn3.click(act_load_strategies, outputs=[dd3, t3_all])
        dd3.change(act_select_strategy, inputs=[dd3], outputs=[p3, t3])

    with gr.Tab("🔍 SHAP 特徵分析"):
        btn4 = gr.Button("📊 產生 SHAP 圖表", variant="primary")
        p4   = gr.Plot()
        t4   = gr.Textbox(label="說明", lines=2)
        btn4.click(act_shap, outputs=[p4, t4])

demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c7cda9563be9bdabe4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
